# IP Anycast Infrastructure Analysis

This notebook maps Tranco domain IP dependencies to Anycast Census/LACeS prefixes and produces domain-level and leading-nameserver anycast summaries.


In [1]:
# Imports and notebook configuration
import inspect
import ipaddress
from datetime import datetime
from pathlib import Path

import pandas as pd
import pytricia
from IPython.display import display

from helpers import census_helper

pd.set_option("display.max_colwidth", None)

PROJECT_DIR = Path.cwd().parent
RESULT_DIR = PROJECT_DIR / "data" / "result"
ANYCAST_CENSUS_DIR = PROJECT_DIR / "data" / "anycast_census"
ANYCAST_RESULT_DIR = RESULT_DIR / "anycast_results"

SNAPSHOT_DATE_STR = "2026-06-01"
SNAPSHOT_DATE = datetime.strptime(SNAPSHOT_DATE_STR, "%Y-%m-%d")

## Domain Anycast Workflow

The first part of the notebook produces a per-domain summary with total IPs, IPv4/IPv6 split, and anycast coverage.


## 1. Load Domain IP Data

Load the IP dependency data produced by the infrastructure analysis. Only the domain and IP columns are needed for the anycast classification step.

In [55]:
ip_prefix_as_df = pd.read_parquet(
    RESULT_DIR / "IP_Prefix_AS_dependency",
    columns=["domain", "IP"],
)

print("Loaded domain rows:", len(ip_prefix_as_df))
display(ip_prefix_as_df.head(2))

Loaded domain rows: 945848


,domain,IP
0,link-988betgcr.pro,"[2a06:98c1:50::ac40:2274, 2803:f800:50::6ca2:c274, 108.162.194.116, 162.159.38.116, 172.64.34.116, 2606:4700:50::a29f:2674, 2803:f800:50::6ca2:c171, 2a06:98c1:50::ac40:2171, 172.64.33.113, 2606:4700:58::adf5:3b71, 108.162.193.113, 173.245.59.113]"
1,desabet.pro,"[2803:f800:50::6ca2:c142, 108.162.193.66, 2a06:98c1:50::ac40:2142, 173.245.59.66, 2606:4700:58::adf5:3b42, 172.64.33.66, 2a06:98c1:50::ac40:22ca, 172.64.34.202, 162.159.38.202, 2606:4700:50::a29f:26ca, 108.162.194.202, 2803:f800:50::6ca2:c2ca]"


## 2. Inspect the Anycast Helper

The helper module downloads LACeS census snapshots and filters them to anycast prefixes. This quick inspection documents the available helper functions before using them below.

In [56]:
for name in dir(census_helper):
    if name.startswith("_"):
        continue

    obj = getattr(census_helper, name)
    if callable(obj):
        try:
            print(name, inspect.signature(obj))
        except ValueError:
            print(name)

datetime
download_date (date_obj: datetime.datetime, version: str) -> pandas.DataFrame
download_latest (version: str) -> pandas.DataFrame
filter_anycast (df: pandas.DataFrame, version: str, confidence: str = 'high') -> pandas.DataFrame
main (args)
store_prefixes_only (ts: datetime.datetime, df: pandas.DataFrame, version: str, output_path: str, confidence: str = 'high') -> None


## 3. Download and Store Anycast Census Data

Download the IPv4 and IPv6 LACeS census snapshots for June 1, 2026. The full census and high-confidence anycast subsets are stored locally so the analysis can be reused without downloading again.

In [57]:
ANYCAST_CENSUS_DIR.mkdir(parents=True, exist_ok=True)

census_v4 = census_helper.download_date(SNAPSHOT_DATE, "v4")
census_v6 = census_helper.download_date(SNAPSHOT_DATE, "v6")

v4_path = ANYCAST_CENSUS_DIR / f"laces_ipv4_{SNAPSHOT_DATE_STR}.parquet"
v6_path = ANYCAST_CENSUS_DIR / f"laces_ipv6_{SNAPSHOT_DATE_STR}.parquet"

census_v4.to_parquet(v4_path, index=False)
census_v6.to_parquet(v6_path, index=False)

print("Saved IPv4 census:", v4_path)
print("Saved IPv6 census:", v6_path)
print("IPv4 census shape:", census_v4.shape)
print("IPv6 census shape:", census_v6.shape)

Saved IPv4 census: /home/semindu/Work/jupyter/ptt/revisit-ptt-in-DNS/data/anycast_census/laces_ipv4_2026-06-01.parquet
Saved IPv6 census: /home/semindu/Work/jupyter/ptt/revisit-ptt-in-DNS/data/anycast_census/laces_ipv6_2026-06-01.parquet
IPv4 census shape: (39562, 10)
IPv6 census shape: (14115, 9)


This cell filters the downloaded census snapshots to high-confidence anycast prefixes and stores the reusable filtered parquet files.


In [2]:
anycast_v4_high = census_helper.filter_anycast(census_v4, "v4", confidence="high")
anycast_v6_high = census_helper.filter_anycast(census_v6, "v6", confidence="high")

anycast_v4_path = ANYCAST_CENSUS_DIR / f"laces_ipv4_anycast_high_{SNAPSHOT_DATE_STR}.parquet"
anycast_v6_path = ANYCAST_CENSUS_DIR / f"laces_ipv6_anycast_high_{SNAPSHOT_DATE_STR}.parquet"

anycast_v4_high.to_parquet(anycast_v4_path, index=False)
anycast_v6_high.to_parquet(anycast_v6_path, index=False)

print("IPv4 high-confidence anycast shape:", anycast_v4_high.shape)
print("IPv6 high-confidence anycast shape:", anycast_v6_high.shape)
display(anycast_v4_high.head())
display(anycast_v6_high.head())

NameError: name 'census_v4' is not defined

## Start from here, If you already have the above cell run at least once

In [3]:
# Reload the saved high-confidence anycast datasets
anycast_v4_path = ANYCAST_CENSUS_DIR / f"laces_ipv4_anycast_high_{SNAPSHOT_DATE_STR}.parquet"
anycast_v6_path = ANYCAST_CENSUS_DIR / f"laces_ipv6_anycast_high_{SNAPSHOT_DATE_STR}.parquet"

for path in (anycast_v4_path, anycast_v6_path):
    if not path.exists():
        raise FileNotFoundError(f"Anycast parquet not found: {path}")

anycast_v4_high = pd.read_parquet(anycast_v4_path)
anycast_v6_high = pd.read_parquet(anycast_v6_path)

print("IPv4 high-confidence anycast shape:", anycast_v4_high.shape)
display(anycast_v4_high.head())

print("IPv6 high-confidence anycast shape:", anycast_v6_high.shape)
display(anycast_v6_high.head())

IPv4 high-confidence anycast shape: (15253, 10)


prefix  AB_ICMPv4  AB_TCPv4  AB_DNSv4  GCD_ICMPv4  GCD_TCPv4  \
0    1.0.0.0/24         30        30        21          75         13   
1    1.1.1.0/24         30        30        22          72         11   
2  1.10.10.0/24          3         0         2           3          0   
3   1.12.0.0/24          5         0         3           5          0   
4  1.12.12.0/24          3         1         0           3          1   

   partial backing_prefix     ASN  \
0    False     1.0.0.0/24   13335   
1    False     1.1.1.0/24   13335   
2    False   1.10.10.0/24  148000   
3    False    1.12.0.0/20  132203   
4    False    1.12.0.0/20  132203   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

IPv6 high-confidence anycast shape: (13258, 9)


,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201::/48,3,2,3,2,1,2001:1201::/48,28498,"[{'city': 'Monterrey', 'country_code': 'MX', 'id': 'MTY', 'lat': 25.7785, 'lon': -100.107}, {'city': 'Querétaro', 'country_code': 'MX', 'id': 'QRO', 'lat': 20.6173, 'lon': -100.186}]"
1,2001:1250:a000::/48,2,0,2,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'country_code': 'MX', 'id': 'MTY', 'lat': 25.7785, 'lon': -100.107}, {'city': 'Houston', 'country_code': 'US', 'id': 'IAH', 'lat': 29.9844, 'lon': -95.3414}]"
2,2001:1250:c000::/48,2,0,2,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'country_code': 'MX', 'id': 'MTY', 'lat': 25.7785, 'lon': -100.107}, {'city': 'Houston', 'country_code': 'US', 'id': 'IAH', 'lat': 29.9844, 'lon': -95.3414}]"
3,2001:129c:205::/48,2,0,0,2,0,2001:129c:205::/48,22356,"[{'city': 'São Paulo', 'country_code': 'BR', 'id': 'SAO', 'lat': -23.5091, 'lon': -46.6378}, {'city': 'Mexico City', 'country_code': 'MX', 'id': 'MEX', 'lat': 19.4363, 'lon': -99.0721}]"
4,2001:12f8:2::/48,3,0,3,3,0,2001:12f8:2::/48,12136,"[{'city': 'Singapore', 'country_code': 'SG', 'id': 'SIN', 'lat': 1.35019, 'lon': 103.994}, {'city': 'São Paulo', 'country_code': 'BR', 'id': 'SAO', 'lat': -23.5091, 'lon': -46.6378}, {'city': 'Frankfurt-am-Main', 'country_code': 'DE', 'id': 'FRA', 'lat': 50.0264, 'lon': 8.54313}]"


## 4. Prepare IP Addresses for Matching

Explode the domain-level IP lists into one row per IP address, classify each IP as IPv4 or IPv6, and keep invalid values separate for checking.

This helper cell classifies each IP string as IPv4, IPv6, or invalid before anycast prefix matching.


In [4]:
def get_ip_version(ip):
    """Return the IP version, or pd.NA when the value is not a valid IP address."""
    try:
        return ipaddress.ip_address(str(ip)).version
    except ValueError:
        return pd.NA

This cell explodes each domain into one row per IP address, separates IPv4 and IPv6 addresses, and reports invalid IP values for quality control.


In [ ]:
df_ip = ip_prefix_as_df.explode("IP").reset_index(drop=True)
df_ip["ip_version"] = df_ip["IP"].apply(get_ip_version)

ipv4_df = df_ip[df_ip["ip_version"] == 4].copy()
ipv6_df = df_ip[df_ip["ip_version"] == 6].copy()
invalid_df = df_ip[df_ip["ip_version"].isna()].copy()

print("Original domain rows:", len(ip_prefix_as_df))
print("Exploded IP rows:", len(df_ip))
print("IPv4 rows:", len(ipv4_df))
print("IPv6 rows:", len(ipv6_df))
print("Invalid IP rows:", len(invalid_df))

display(df_ip.head(20))

Original domain rows: 945848
Exploded IP rows: 9345932
IPv4 rows: 5281270
IPv6 rows: 4064662
Invalid IP rows: 0


,domain,IP,ip_version
0,link-988betgcr.pro,2a06:98c1:50::ac40:2274,6
1,link-988betgcr.pro,2803:f800:50::6ca2:c274,6
2,link-988betgcr.pro,108.162.194.116,4
3,link-988betgcr.pro,162.159.38.116,4
4,link-988betgcr.pro,172.64.34.116,4
5,link-988betgcr.pro,2606:4700:50::a29f:2674,6
6,link-988betgcr.pro,2803:f800:50::6ca2:c171,6
7,link-988betgcr.pro,2a06:98c1:50::ac40:2171,6
8,link-988betgcr.pro,172.64.33.113,4
9,link-988betgcr.pro,2606:4700:58::adf5:3b71,6


## 5. Build Prefix Tries

PyTricia stores anycast prefixes in a Patricia trie, which makes longest-prefix matching efficient for both IPv4 and IPv6 addresses.

In [60]:
def build_prefix_trie(anycast_df, prefix_col, ip_version):
    trie = pytricia.PyTricia(32 if ip_version == 4 else 128)

    for prefix in anycast_df[prefix_col].dropna().astype(str).unique():
        trie[prefix] = prefix

    return trie


trie_v4 = build_prefix_trie(anycast_v4_high, "prefix", ip_version=4)
trie_v6 = build_prefix_trie(anycast_v6_high, "prefix", ip_version=6)

print("IPv4 trie prefixes:", len(trie_v4))
print("IPv6 trie prefixes:", len(trie_v6))

IPv4 trie prefixes: 15253
IPv6 trie prefixes: 13258


## 6. Match IPs to Anycast Prefixes

Match each unique IP address once, then merge the classification back to the exploded domain-IP table.

In [61]:
def match_anycast_ips(unique_ip_df, trie):
    results = []

    for ip in unique_ip_df["IP"]:
        ip_str = str(ip)

        try:
            matched_prefix = trie.get(ip_str)
        except Exception:
            matched_prefix = None

        results.append({
            "IP": ip_str,
            "is_anycast": matched_prefix is not None,
            "matched_anycast_prefix": matched_prefix,
        })

    return pd.DataFrame(results)


unique_ipv4_df = ipv4_df[["IP"]].drop_duplicates().reset_index(drop=True)
unique_ipv6_df = ipv6_df[["IP"]].drop_duplicates().reset_index(drop=True)

ipv4_anycast_result = match_anycast_ips(unique_ipv4_df, trie_v4)
ipv6_anycast_result = match_anycast_ips(unique_ipv6_df, trie_v6)

print("IPv4 rows:", len(ipv4_df))
print("Unique IPv4:", len(unique_ipv4_df))
print(ipv4_anycast_result["is_anycast"].value_counts(dropna=False))

print("\nIPv6 rows:", len(ipv6_df))
print("Unique IPv6:", len(unique_ipv6_df))
print(ipv6_anycast_result["is_anycast"].value_counts(dropna=False))

IPv4 rows: 5281270
Unique IPv4: 186630
is_anycast
False    172949
True      13681
Name: count, dtype: int64

IPv6 rows: 4064662
Unique IPv6: 35212
is_anycast
False    24228
True     10984
Name: count, dtype: int64


This cell combines IPv4 and IPv6 anycast matches, joins them back to the exploded domain-IP table, and checks the classification coverage.


In [ ]:
ip_anycast_result = pd.concat(
    [ipv4_anycast_result, ipv6_anycast_result],
    ignore_index=True,
)

df_ip_anycast = df_ip.merge(
    ip_anycast_result,
    on="IP",
    how="left",
)

print("Domain-IP rows with anycast classification:", df_ip_anycast.shape)
print(df_ip_anycast["is_anycast"].value_counts(dropna=False))
print("Missing anycast classification:", df_ip_anycast["is_anycast"].isna().sum())

display(df_ip_anycast.head())

## 7. Summarize Anycast Coverage by Domain

Deduplicate domain-IP pairs before grouping so repeated IPs do not inflate the per-domain counts.

In [62]:
dedup_domain_ip = df_ip_anycast.drop_duplicates(["domain", "IP"])

domain_anycast_summary = (
    dedup_domain_ip
    .groupby("domain")
    .agg(
        total_ips=("IP", "nunique"),
        ipv4_count=("ip_version", lambda x: (x == 4).sum()),
        ipv6_count=("ip_version", lambda x: (x == 6).sum()),
        anycast_ips=("is_anycast", "sum"),
        ipv4_anycast=("is_anycast", lambda x: (
            dedup_domain_ip.loc[x.index, "ip_version"].eq(4) & x
        ).sum()),
        ipv6_anycast=("is_anycast", lambda x: (
            dedup_domain_ip.loc[x.index, "ip_version"].eq(6) & x
        ).sum()),
        has_anycast=("is_anycast", "any"),
    )
    .reset_index()
)

display(domain_anycast_summary)

,domain,total_ips,ipv4_count,ipv6_count,anycast_ips,ipv4_anycast,ipv6_anycast,has_anycast
0,0-24.jp,4,4,0,0,0,0,False
1,0-55.ru,2,2,0,0,0,0,False
2,0-6.com,24,22,2,4,2,2,True
3,0-8-0-a.com,12,6,6,12,6,6,True
4,0-c-e-u.com,12,6,6,12,6,6,True
...,...,...,...,...,...,...,...,...
945843,zzzzsj.co,12,6,6,12,6,6,True
945844,zzzzsj.com,4,2,2,4,2,2,True
945845,zzzzsj.top,12,6,6,12,6,6,True
945846,zzzzsj.vip,12,6,6,12,6,6,True


## 8. Save the Domain Summary

Write the domain-level anycast summary to parquet for reuse in later notebooks or reporting steps.

In [63]:
ANYCAST_RESULT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = ANYCAST_RESULT_DIR / "domain_anycast_summary.parquet"
domain_anycast_summary.to_parquet(summary_path, index=False)

print("Saved domain anycast summary:", summary_path)

Saved domain anycast summary: /home/semindu/Work/jupyter/ptt/revisit-ptt-in-DNS/data/result/anycast_results/domain_anycast_summary.parquet


## Leading Nameserver Anycast Analysis

This section studies the nameservers that account for the first 10 percent of observed nameserver frequency and classifies their IP infrastructure as anycast, mixed, or unicast.


This cell loads the leading nameserver IP table, explodes nameserver-IP pairs, matches them to the anycast prefix tries, and summarizes anycast coverage per nameserver.


In [13]:
# Load nameserver IP data saved by the infrastructure notebook
nameserver_ip_path = (
    RESULT_DIR / "leading_servers" / "tranco_nameserver_ip_data.parquet"
)
nameserver_ip_df = pd.read_parquet(nameserver_ip_path)

# Create one row for each nameserver-IP pair
nameserver_ip_exploded = (
    nameserver_ip_df.explode("IP")
    .dropna(subset=["IP"])
    .reset_index(drop=True)
)
nameserver_ip_exploded["IP"] = nameserver_ip_exploded["IP"].astype(str)
nameserver_ip_exploded["ip_version"] = (
    nameserver_ip_exploded["IP"].apply(get_ip_version)
)

invalid_nameserver_ips = nameserver_ip_exploded[
    nameserver_ip_exploded["ip_version"].isna()
].copy()
valid_nameserver_ips = nameserver_ip_exploded[
    nameserver_ip_exploded["ip_version"].notna()
].copy()

# Match each unique address against the appropriate anycast prefix trie
unique_ns_ipv4 = (
    valid_nameserver_ips.loc[valid_nameserver_ips["ip_version"] == 4, ["IP"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_ns_ipv6 = (
    valid_nameserver_ips.loc[valid_nameserver_ips["ip_version"] == 6, ["IP"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

ns_anycast_matches = pd.concat(
    [
        match_anycast_ips(unique_ns_ipv4, trie_v4),
        match_anycast_ips(unique_ns_ipv6, trie_v6),
    ],
    ignore_index=True,
)

nameserver_ip_anycast_df = valid_nameserver_ips.merge(
    ns_anycast_matches,
    on="IP",
    how="left",
)
nameserver_ip_anycast_df["is_anycast"] = (
    nameserver_ip_anycast_df["is_anycast"].fillna(False).astype(bool)
)

# One summary row per nameserver
dedup_ns_ip = nameserver_ip_anycast_df.drop_duplicates(
    ["NameServer", "IP"]
).copy()
dedup_ns_ip["ipv4_anycast"] = (
    dedup_ns_ip["is_anycast"] & dedup_ns_ip["ip_version"].eq(4)
)
dedup_ns_ip["ipv6_anycast"] = (
    dedup_ns_ip["is_anycast"] & dedup_ns_ip["ip_version"].eq(6)
)

nameserver_anycast_summary = (
    dedup_ns_ip.groupby("NameServer", as_index=False)
    .agg(
        total_ips=("IP", "nunique"),
        anycast_ips=("is_anycast", "sum"),
        ipv4_anycast=("ipv4_anycast", "sum"),
        ipv6_anycast=("ipv6_anycast", "sum"),
        has_anycast=("is_anycast", "any"),
    )
    .sort_values(["has_anycast", "anycast_ips"], ascending=[False, False])
    .reset_index(drop=True)
)

print("Loaded nameservers:", len(nameserver_ip_df))
print("Exploded valid IP rows:", len(valid_nameserver_ips))
print("Invalid IP rows:", len(invalid_nameserver_ips))
print("Nameservers with anycast:", nameserver_anycast_summary["has_anycast"].sum())

display(nameserver_ip_anycast_df.head(20))
display(nameserver_anycast_summary)

Loaded nameservers: 80
Exploded valid IP rows: 258
Invalid IP rows: 0
Nameservers with anycast: 56


,NameServer,IPv4_count,IP_COUNT,IP,ip_version,is_anycast,matched_anycast_prefix
0,vip3.alidns.com,10,11,47.116.84.178,4,False,NaN
1,vip3.alidns.com,10,11,8.212.93.3,4,False,NaN
2,vip3.alidns.com,10,11,170.33.40.136,4,False,NaN
3,vip3.alidns.com,10,11,2408:4009:500::3,6,True,2408:4009:500::/48
4,vip3.alidns.com,10,11,39.103.26.212,4,False,NaN
5,vip3.alidns.com,10,11,121.40.6.163,4,False,NaN
6,vip3.alidns.com,10,11,170.33.80.10,4,False,NaN
7,vip3.alidns.com,10,11,8.129.152.245,4,False,NaN
8,vip3.alidns.com,10,11,170.33.73.26,4,False,NaN
9,vip3.alidns.com,10,11,170.33.32.210,4,False,NaN


,NameServer,total_ips,anycast_ips,ipv4_anycast,ipv6_anycast,has_anycast
0,ns-com.ui-dns.com,8,8,4,4,True
1,andy.ns.cloudflare.com,6,6,3,3,True
2,darl.ns.cloudflare.com,6,6,3,3,True
3,dora.ns.cloudflare.com,6,6,3,3,True
4,nelly.ns.cloudflare.com,6,6,3,3,True
...,...,...,...,...,...,...
75,ns3.second-ns.de,2,0,0,0,False
76,ns4-cloud.nic.ru,8,0,0,0,False
77,ns4-l2.nic.ru,1,0,0,0,False
78,ns8-cloud.nic.ru,8,0,0,0,False


This cell saves the leading-nameserver anycast summary so the final analysis notebook can compare major nameserver infrastructure by anycast deployment type.


In [14]:
# Save the leading nameserver anycast summary
ANYCAST_RESULT_DIR.mkdir(parents=True, exist_ok=True)

leading_nameserver_path = ANYCAST_RESULT_DIR / "leading_nameservers.parquet"
nameserver_anycast_summary.to_parquet(
    leading_nameserver_path,
    index=False,
)

print(f"Saved {len(nameserver_anycast_summary):,} rows to {leading_nameserver_path}")

Saved 80 rows to /home/semindu/Work/jupyter/ptt/revisit-ptt-in-DNS/data/result/anycast_results/leading_nameservers.parquet
